In [58]:
import pandas as pd

In [59]:
low_confidence = pd.read_json(r"/home/kaka/tcc/ml/low_confidence_tweets.json")
low_confidence.head()

In [3]:
import os 

# get files from dir
def get_files_from_dir(dir_path):
    files = []
    for root, dirs, filenames in os.walk(dir_path):
        for filename in filenames:
            if filename.endswith('.csv'):
                files.append(os.path.join(root, filename))
    return files
# merge multiple csv files into one
def merge_csv_files(file_list, output_file):
    # Read the first file to initialize the DataFrame
    df = pd.read_csv(file_list[0])

    # Loop through the rest of the files and append them to the DataFrame
    for file in file_list[1:]:
        temp_df = pd.read_csv(file)
        df = pd.concat([df, temp_df], ignore_index=True)

    # Save the merged DataFrame to a new CSV file
    df.to_csv(output_file, index=False)

files = get_files_from_dir("/home/kaka/tcc/ml/tweets_rotulados")

merge_csv_files(files, "/home/kaka/tcc/ml/tweets_rotulados/tweets_rotulados.csv")

In [64]:
ground_truth = pd.read_csv(r"/home/kaka/tcc/ml/tweets_rotulados/tweets_rotulados.csv")


In [65]:
# combine tags columns into one, do a regex rule to capture the tags columns (because i have 44 tags columns)
import re
def combine_tags_columns(df):
    # Create a new column 'tags' : 'tags[0]', 'tags[1]', 'tags[2]', 'tags[3]'... until 'tags[44]'
    df['tags'] = df.apply(lambda row: [row[col] for col in df.columns if re.match(r'tags\[\d+\]', col)], axis=1)
    # Remove empty tags
    df['tags'] = df['tags'].apply(lambda x: [tag for tag in x if pd.notna(tag)])
    

    # Remove the original tags columns
    tags_columns = [col for col in df.columns if re.match(r'tags\[\d+\]', col)]
    df.drop(columns=tags_columns, inplace=True)
    return df
ground_truth = combine_tags_columns(ground_truth)

In [66]:
# transform column tags into a string
ground_truth['tags'] = ground_truth['tags'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

In [67]:
ground_truth['tags'].str.lower()

# get unique tags
unique_tags = set()
for tags in ground_truth['tags']:
    if isinstance(tags, str):
        tags_list = tags.split(', ')
        unique_tags.update(tags_list)
unique_tags = list(unique_tags)
print(unique_tags)


['', 'corrupto', 'destrói', 'canibalismo', 'carne', 'romeu zema', 'bolsonaro', 'noticia verdadeira', 'STF', 'sem provas', 'TSE', 'mg', 'satanico', 'Minas Gerais', 'Miriam Cordeiro', 'internacional', 'bolsoaro', 'oposição', 'Governador', 'neto', 'nine', 'proibiu', 'canibal', 'presidente', 'abortista', 'ameaça', 'minas', 'Minas', 'interessante', 'comunismo', 'mineiro', 'Canibal', 'governador', 'cachaceiro', 'ameça', 'ex-presidiário', 'jair', 'lula', 'ameaçou', 'contra', 'MG', 'verba', 'satânico', 'reportagem', 'federal', 'Zema', 'stf', 'humana', 'ameaçando', 'Lula', 'minas gerais', 'Rouanet', 'brasil', 'Ameaça', 'Canibalismo', 'estado', 'ameaçar', 'zema', 'mídia', 'erro', 'ministro', '9 dedos', 'Bolsonaro', 'the guardian', 'Aborto', 'ditatoriais', 'jairbolsonaro']


In [68]:
ground_truth

In [69]:
# get tweets with label 'fake'

ground_truth_fake = ground_truth.query("label == 'fake'")
ids_fake_label = ground_truth_fake['tweet_id'].tolist()
print('Tweets com fake news: ' + str(len(ids_fake_label)))

# get tweets with label 'true'
ground_truth_true = ground_truth.query("label == 'true'")
ids_true_label = ground_truth_true['tweet_id'].tolist()
print('Tweets com true news: ' + str(len(ids_true_label)))

# get tweets with label 'irrelevant'
ground_truth_irrelevant = ground_truth.query("label == 'irrelevant'")
ids_irrelevant_label = ground_truth_irrelevant['tweet_id'].tolist()
print('Tweets irrelevantes: ' + str(len(ids_irrelevant_label)))

Tweets com fake news: 44
Tweets com true news: 4
Tweets irrelevantes: 2


In [70]:
# update the label of low_confidence tweets with the label of the ground_truth
low_confidence['label'] = low_confidence['tweet_id'].apply(lambda x: 'fake' if x in ids_fake_label else ('true' if x in ids_true_label else ('irrelevant' if x in ids_irrelevant_label else None)))
low_confidence['label'].value_counts()

label
fake          26
true           3
irrelevant     1
Name: count, dtype: int64

In [71]:
# change label for fake for 1 and true for 0
ground_truth['label'] = ground_truth['label'].replace({'fake': 1, 'true': 0})
ground_truth['label'].value_counts()

# drop irrelevant tweets
ground_truth = ground_truth.query("label != 'irrelevant'")
ground_truth['label'].value_counts()


label
1    44
0     4
Name: count, dtype: int64

In [72]:
# save the low_confidence tweets with the label
# filter rows that have label == None 
manual_label = ground_truth[ground_truth['label'].notna()]

In [73]:
manual_label.to_csv(r"/home/kaka/tcc/ml/tweets_rotulados/tweets_rotulados_normalizados.csv", index=False)

In [61]:
low_confidence